In [0]:
from pyspark.sql.functions import *

In [0]:
storage_account= "storageretaillakehouse"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    "<YOUR KEY>"
)


In [0]:
fact_sales_path = f"abfss://gold@storageretaillakehouse.dfs.core.windows.net/facts/fact_sales"

In [0]:
fs_df = spark.read.format("delta").load(fact_sales_path)

fs_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- customer_key: long (nullable = true)
 |-- product_key: long (nullable = true)
 |-- date_key: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- total_payment_value: double (nullable = true)
 |-- total_installments: long (nullable = true)
 |-- sale_key: long (nullable = true)



In [0]:
sales_summary = fs_df.agg(
    countDistinct("order_id")
            .alias("total_orders"),

    sum("total_payment_value")
            .alias("total_revenue")
)

In [0]:
sales_summary = sales_summary.withColumn(
    "avg_order_value",
    col("total_revenue") / col("total_orders")
)

In [0]:
display(sales_summary)

total_orders,total_revenue,avg_order_value
98666,2.030813470999948E7,205.82708035188898


In [0]:
from common_scripts.dq_framework import *

run_dq_check(
    sales_summary,
    "sales_summary",
    "null_total_revenue",
    col("total_revenue").isNull(),
    severity="critical"
)

run_dq_check(
    sales_summary,
    "sales_summary",
    "null_total_orders",
    col("total_orders").isNull(),
    severity="critical"
)

[CRITICAL] sales_summary | null_total_revenue: 0
[CRITICAL] sales_summary | null_total_orders: 0


0

In [0]:
dq_df = dq_df_from_dq_results(spark, dq_results)

dq_path = f"abfss://audit@{storage_account}.dfs.core.windows.net/gold_dq_logs/kpi_sales_summary"

dq_df.write.format('delta').mode('append').save(dq_path)

In [0]:
sales_summary_path =  f"abfss://gold@{storage_account}.dfs.core.windows.net/kpi/kpi_sales_summary"


sales_summary.write.format('delta').mode('overwrite').save(sales_summary_path)